In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pickle
from met_brewer import met_brew
import matplotlib.gridspec as gridspec
import os

%matplotlib inline


In [ ]:
# plot function

def plot_multiruns(experiment_folder, label, metric_file, ax, color):

    loss_all_runs = []
    
    # Traverse through each run folder inside the experiment folder
    for run_folder in os.listdir(experiment_folder):
        run_path = os.path.join(experiment_folder, run_folder)
        if os.path.isdir(run_path):
            train_info_folder = os.path.join(run_path, 'train_info')
                # Look for .pkl files inside the train_info folder
            if os.path.exists(train_info_folder):
                file = metric_file
                file_path = os.path.join(train_info_folder, file) 
                # Load the .pkl file and extract the loss
                with open(file_path, 'rb') as f:
                    loss_all_runs.append(pickle.load(f))
                # Convert list of loss into a numpy array (padding shorter runs with NaN)
        
    max_epochs = max(len(run_loss) for run_loss in loss_all_runs)
    losses_padded = np.array([np.pad(run_loss, (0, max_epochs - len(run_loss)), 'constant', constant_values=np.nan)
                                for run_loss in loss_all_runs])

    # Compute mean and variance, ignoring NaNs
    mean_loss = np.nanmean(losses_padded, axis=0)

    lower_percentile = np.nanpercentile(losses_padded, 5, axis=0)
    upper_percentile = np.nanpercentile(losses_padded, 90, axis=0)
    

    std_dev = np.nanstd(losses_padded, axis=0)
    num_samples = np.sum(~np.isnan(losses_padded), axis=0)  # Count non-NaN samples for each epoch

    # Compute the 95% CI (mean ± 1.96 * std_err)
    std_error = std_dev / np.sqrt(num_samples)
    lower_ci = mean_loss - 1.96 * std_error
    upper_ci = mean_loss + 1.96 * std_error
    lower_ci = np.clip(lower_ci, 0, None)  # Clip values to a minimum of 0
    

    # Plot the average loss with variance as a shaded area
    epochs = np.arange(0, max_epochs * 25, 25)

    ax.plot(epochs, mean_loss, label=label, c=color, linewidth=0.5)
    ax.fill_between(epochs, lower_ci, upper_ci, color=color, alpha=0.4, linewidth=0.5)
    ax.legend()

In [ ]:
# graphical properties
plt.rcParams["axes.edgecolor"] = "k"
plt.rcParams["axes.facecolor"] = "w"
plt.rcParams["axes.linewidth"] = "0.8"
plt.rcParams.update({'font.size': 6})
plt.rcParams['savefig.dpi'] = 300

plt.rcParams['pdf.fonttype'] = 42 # prepare as vector graphic
plt.rcParams['ps.fonttype'] = 42
plt.rcParams.update({'axes.titlesize': 7})
plt.rcParams.update({'legend.fontsize': 7})
plt.rcParams["font.family"] = "Arial"

In [ ]:
# Create Figure and axs
cm = 1/2.54  # centimeters in inches
fig1 = plt.figure(figsize=(11.4*cm, 3.3*cm))
gs1 = gridspec.GridSpec(1, 3)

#fig1.subplots_adjust(hspace=1.5, wspace=0.9)



In [ ]:
#delayed match
ax1 = fig1.add_subplot(gs1[0])

colors = ["#C20017", "#cac100ff", "#23AA00"]
comparison_experiment_paths = [r"C:\Users\j1559\Documents\Tuebingen\SS_24\MasterThesis\neuromodRNNs\neuromodRNN\outputs\pattern_generation\Fig_1\BPTT_lr_010",
                               r"C:\Users\j1559\Documents\Tuebingen\SS_24\MasterThesis\neuromodRNNs\neuromodRNN\outputs\pattern_generation\Fig_1\e_prop_hardcoded_lr_010",
                               r"C:\Users\j1559\Documents\Tuebingen\SS_24\MasterThesis\neuromodRNNs\neuromodRNN\outputs\pattern_generation\Fig_1\diffusion_k_075_lr_010",
                              ]
            
                              

labels = ["BPTT", "E-prop", "Diffusion"]

for experiment_path, label, color in zip(comparison_experiment_paths, labels, colors):
    plot_multiruns(experiment_folder=experiment_path, label=label,metric_file="nMSE_eval.pkl", ax=ax1, color=color)

#ax1.legend(loc='lower left')
#ax1.legend().set_visible(False)
ax1.legend(frameon=False)  # Removes the box around the legend
ax1.set_ylabel("nMSE", labelpad=2)
ax1.set_xlabel("Iterations", labelpad=2)
ax1.set_title("Pattern generation")
ax1.spines['right'].set_visible(False)
ax1.spines['top'].set_visible(False)
ax1.set_yticks([0.05, 0.15, 0.25, 0.35])

ax1.set_ylim(0, 0.4)


In [ ]:
ax2 = fig1.add_subplot(gs1[1])
colors = ["#C20017", "#cac100ff", "#23AA00"]
comparison_experiment_paths = [r"C:\Users\j1559\Documents\Tuebingen\SS_24\MasterThesis\neuromodRNNs\neuromodRNN\outputs\delayed_match\Fig_1\BPTT_lr_0050_c_reg_0_50",
                               r"C:\Users\j1559\Documents\Tuebingen\SS_24\MasterThesis\neuromodRNNs\neuromodRNN\outputs\delayed_match\Fig_1\mean_e_prop_lr_0050_c_reg_0_10",
                               r"C:\Users\j1559\Documents\Tuebingen\SS_24\MasterThesis\neuromodRNNs\neuromodRNN\outputs\delayed_match\Fig_1\diffusion_k_075_lr_0050_c_reg_0_01",
                              ]
            
                              

labels = ["BPTT", "E-prop", "Diffusion"]

for experiment_path, label, color in zip(comparison_experiment_paths, labels, colors):
    plot_multiruns(experiment_folder=experiment_path, label=label,metric_file="loss_eval.pkl", ax=ax2, color=color)

#ax2.legend(loc='lower left')
ax2.legend().set_visible(False)
ax2.set_ylabel("Cross-Entropy", labelpad=2)
#ax2.set_xlabel("Iterations")
ax2.set_title("Delayed match to sample")
ax2.spines['right'].set_visible(False)
ax2.spines['top'].set_visible(False)
ax2.set_yticks([ 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7])
ax2.set_ylim(0, 0.735)


In [ ]:
ax3 = fig1.add_subplot(gs1[2])

colors = ["#C20017", "#cac100ff", "#23AA00"]
comparison_experiment_paths = [r"C:\Users\j1559\Documents\Tuebingen\SS_24\MasterThesis\neuromodRNNs\neuromodRNN\outputs\cue_accumulation\Fig_1\bptt_lr_005_creg_005",
                               r"C:\Users\j1559\Documents\Tuebingen\SS_24\MasterThesis\neuromodRNNs\neuromodRNN\outputs\cue_accumulation\Fig_1\e_prop_lr_005_creg_005",
                               r"C:\Users\j1559\Documents\Tuebingen\SS_24\MasterThesis\neuromodRNNs\neuromodRNN\outputs\cue_accumulation\Fig_1\diffusion_k_075_lr_005_creg_005",
                              ]
            
                              

labels = ["BPTT", "E-prop", "Diffusion"]

for experiment_path, label, color in zip(comparison_experiment_paths, labels, colors):
    plot_multiruns(experiment_folder=experiment_path, label=label,metric_file="loss_eval.pkl", ax=ax3, color=color)

#ax3.legend(loc='lower left')
#ax3.set_ylabel("Cross-Entropy")
#ax3.set_xlabel("Iterations")
ax3.legend().set_visible(False)
ax3.set_title("Cue accumulation")
# ax3.set_ylabel("Cross-Entropy", labelpad=2)
ax3.spines['right'].set_visible(False)
ax3.spines['top'].set_visible(False)
ax3.set_yticks([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7])
ax3.set_ylim(0, 0.735)


In [ ]:
#fig1.tight_layout()
fig1.subplots_adjust(left=0.09, right=0.988, top=0.85, bottom=0.23, wspace=0.35, hspace=0.05)
fig1.savefig("Fig2.pdf", format="pdf")
plt.show()